<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/05_embeddings_qdrant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Embeddings and Qdrant | Phase 6.1

## Notebook 05

This notebook generates dense vector embeddings for MetricGuard's fully
enriched retrieval chunks and stores them in Qdrant.

Production knowledge preparation lives in:

- `src/metricguard/ingestion/`
- `src/metricguard/chunking/`
- `src/metricguard/metadata/`
- `src/metricguard/governance/`
- `src/metricguard/lineage/`

Notebook 05 imports and executes those production components.

Pipeline:

raw sources
→ parsing
→ chunking
→ governance
→ lineage / impact
→ embeddings
→ Qdrant
→ semantic retrieval

Ground-truth evaluation files are never embedded or stored in Qdrant.

In [1]:
from pathlib import Path
import shutil
import subprocess

GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/metricguard-ai.git"
REPO_DIR = Path("/content/metricguard-ai")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True,
)

print("Repository cloned:", REPO_DIR)

Repository cloned: /content/metricguard-ai


In [2]:
%pip install -q -e "/content/metricguard-ai" sentence-transformers qdrant-client

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 7.9 MB/s eta 0:00:00
  Building editable for metricguard-ai (pyproject.toml) ... done


In [3]:
import sys

SOURCE_DIR = REPO_DIR / "src"

if str(SOURCE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_DIR))

print("MetricGuard source:", SOURCE_DIR)

MetricGuard source: /content/metricguard-ai/src


In [4]:
from datetime import date
import json
from pathlib import Path
import uuid

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client import models

from metricguard.lineage.enrichment_pipeline import (
    run_full_knowledge_enrichment,
)

In [5]:
run_full_knowledge_enrichment(
    REPO_DIR,
    as_of_date=date(2026, 8, 18),
)

METRICGUARD FULL KNOWLEDGE ENRICHMENT REPORT
Parsed documents      : 55
Final chunks          : 167
Metric-aware chunks   : 97
Lineage-aware chunks  : 90
Lineage graph nodes   : 33
Lineage graph edges   : 30
Freshness as-of       : 2026-08-18
Ground truth          : excluded
Embedding readiness   : YES


In [6]:
CHUNKS_PATH = (
    REPO_DIR
    / "data"
    / "processed"
    / "fully_enriched_chunks.jsonl"
)

print("Chunks file exists:", CHUNKS_PATH.exists())
print("Path:", CHUNKS_PATH)

Chunks file exists: True
Path: /content/metricguard-ai/data/processed/fully_enriched_chunks.jsonl


In [7]:
chunks = []

with CHUNKS_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    for line in file:
        chunks.append(
            json.loads(line)
        )

print("Loaded chunks:", len(chunks))

Loaded chunks: 167


In [8]:
chunks[0]

{'chunk_id': 'active_customer_review-1f2c34aa6c80-chunk-0000',
 'content': '# Active Customer Definition Review  \nauthor: Rohan Mehta\nteam: Customer Analytics\ndate: 2026-03-10\nrelated_metric: active_customers  \nThe enterprise Active Customer definition changed on March 1, 2026.  \nThe approved definition now requires at least one successfully paid order\nduring the previous 30 days.  \nGrowth reporting currently continues to count identified customers with recent\ndigital activity.  \nThat measure remains useful for engagement analysis but should not be treated\nas equivalent to the enterprise Active Customers KPI.  \nRecommendation:  \nEither migrate the Growth dashboard to Active Customers v2 or rename the\nexisting measure to Digital Active Customers.',
 'metadata': {'document_id': 'active_customer_review-1f2c34aa6c80',
  'source_path': 'data/raw/analyst_notes/active_customer_review.md',
  'file_name': 'active_customer_review.md',
  'source_type': 'markdown',
  'asset_type': 'a

In [9]:
assert not any(
    "ground_truth"
    in chunk["metadata"].get(
        "source_path",
        ""
    )
    for chunk in chunks
)

print("✅ Ground truth excluded from embeddings.")

✅ Ground truth excluded from embeddings.


In [10]:
EMBEDDING_MODEL = (
    "sentence-transformers/all-mpnet-base-v2"
)

print(
    "Embedding model:",
    EMBEDDING_MODEL
)

Embedding model: sentence-transformers/all-mpnet-base-v2


In [11]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [13]:
VECTOR_SIZE = (
    embedding_model
    .get_embedding_dimension()
)

print(
    "Embedding dimension:",
    VECTOR_SIZE
)

Embedding dimension: 768


In [14]:
def build_embedding_text(
    chunk: dict,
) -> str:

    metadata = chunk["metadata"]

    context_parts = [
        f"Source type: {metadata.get('source_type')}",
        f"Asset type: {metadata.get('asset_type')}",
        f"File: {metadata.get('file_name')}",
    ]

    if metadata.get("metric_name"):
        context_parts.append(
            f"Metric: {metadata['metric_name']}"
        )

    if metadata.get("observed_version"):
        context_parts.append(
            "Observed version: "
            f"{metadata['observed_version']}"
        )

    if metadata.get("authoritative_version"):
        context_parts.append(
            "Authoritative version: "
            f"{metadata['authoritative_version']}"
        )

    if metadata.get("version_relation"):
        context_parts.append(
            "Version relation: "
            f"{metadata['version_relation']}"
        )

    if metadata.get("freshness_status"):
        context_parts.append(
            "Freshness: "
            f"{metadata['freshness_status']}"
        )

    context = "\n".join(
        context_parts
    )

    return (
        f"{context}\n\n"
        f"{chunk['content']}"
    )

In [15]:
example_text = build_embedding_text(
    chunks[0]
)

print(example_text)

Source type: markdown
Asset type: analyst_note
File: active_customer_review.md
Metric: active_customers
Authoritative version: v2
Version relation: unknown_observed_version
Freshness: unknown

# Active Customer Definition Review  
author: Rohan Mehta
team: Customer Analytics
date: 2026-03-10
related_metric: active_customers  
The enterprise Active Customer definition changed on March 1, 2026.  
The approved definition now requires at least one successfully paid order
during the previous 30 days.  
Growth reporting currently continues to count identified customers with recent
digital activity.  
That measure remains useful for engagement analysis but should not be treated
as equivalent to the enterprise Active Customers KPI.  
Recommendation:  
Either migrate the Growth dashboard to Active Customers v2 or rename the
existing measure to Digital Active Customers.


In [16]:
embedding_texts = [
    build_embedding_text(chunk)
    for chunk in chunks
]

print(
    "Texts prepared:",
    len(embedding_texts)
)

Texts prepared: 167


In [17]:
len(chunks)

167

## Generating Dense Embeddings

In [18]:
embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [19]:
print(
    "Embedding matrix shape:",
    embeddings.shape
)

Embedding matrix shape: (167, 768)


In [20]:
assert len(embeddings) == len(chunks)

assert embeddings.shape[1] == VECTOR_SIZE

assert np.isfinite(
    embeddings
).all()

print(
    "✅ Embedding matrix validated."
)

✅ Embedding matrix validated.


## Creating Local Qdrant Database

In [21]:
QDRANT_PATH = (
    "/content/qdrant_metricguard"
)

qdrant_client = QdrantClient(
    path=QDRANT_PATH
)

print(
    "Qdrant local path:",
    QDRANT_PATH
)

Qdrant local path: /content/qdrant_metricguard


In [22]:
COLLECTION_NAME = "metricguard_dense_v1"

print(
    "Collection:",
    COLLECTION_NAME
)

Collection: metricguard_dense_v1


In [23]:
if qdrant_client.collection_exists(
    collection_name=COLLECTION_NAME
):
    qdrant_client.delete_collection(
        collection_name=COLLECTION_NAME
    )

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,
    ),
)

print(
    "✅ Qdrant collection created."
)

✅ Qdrant collection created.


In [24]:
def chunk_id_to_uuid(
    chunk_id: str,
) -> str:

    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            chunk_id,
        )
    )

In [25]:
print(
    chunks[0]["chunk_id"]
)

print(
    chunk_id_to_uuid(
        chunks[0]["chunk_id"]
    )
)

active_customer_review-1f2c34aa6c80-chunk-0000
914a11d8-610c-58f2-8cd6-9f16944a10df


In [26]:
def build_qdrant_payload(
    chunk: dict,
) -> dict:

    return {
        "chunk_id": chunk["chunk_id"],
        "content": chunk["content"],
        "embedding_model":
            EMBEDDING_MODEL,
        **chunk["metadata"],
    }

In [27]:
build_qdrant_payload(
    chunks[0]
)

{'chunk_id': 'active_customer_review-1f2c34aa6c80-chunk-0000',
 'content': '# Active Customer Definition Review  \nauthor: Rohan Mehta\nteam: Customer Analytics\ndate: 2026-03-10\nrelated_metric: active_customers  \nThe enterprise Active Customer definition changed on March 1, 2026.  \nThe approved definition now requires at least one successfully paid order\nduring the previous 30 days.  \nGrowth reporting currently continues to count identified customers with recent\ndigital activity.  \nThat measure remains useful for engagement analysis but should not be treated\nas equivalent to the enterprise Active Customers KPI.  \nRecommendation:  \nEither migrate the Growth dashboard to Active Customers v2 or rename the\nexisting measure to Digital Active Customers.',
 'embedding_model': 'sentence-transformers/all-mpnet-base-v2',
 'document_id': 'active_customer_review-1f2c34aa6c80',
 'source_path': 'data/raw/analyst_notes/active_customer_review.md',
 'file_name': 'active_customer_review.md',

In [28]:
points = []

for chunk, vector in zip(
    chunks,
    embeddings,
):

    points.append(
        models.PointStruct(
            id=chunk_id_to_uuid(
                chunk["chunk_id"]
            ),
            vector=vector.tolist(),
            payload=build_qdrant_payload(
                chunk
            ),
        )
    )

print(
    "Qdrant points prepared:",
    len(points)
)

Qdrant points prepared: 167


In [29]:
BATCH_SIZE = 64

for start in range(
    0,
    len(points),
    BATCH_SIZE,
):

    batch = points[
        start:
        start + BATCH_SIZE
    ]

    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True,
    )

print(
    "✅ All points uploaded to Qdrant."
)

✅ All points uploaded to Qdrant.


In [30]:
collection_info = (
    qdrant_client
    .get_collection(
        collection_name=COLLECTION_NAME
    )
)

print(
    "Qdrant points:",
    collection_info.points_count
)

print(
    "Expected:",
    len(chunks)
)

Qdrant points: 167
Expected: 167


In [31]:
assert (
    collection_info.points_count
    == len(chunks)
)

print(
    "✅ Qdrant point count validated."
)

✅ Qdrant point count validated.


## Creating query embedding helper

In [32]:
def embed_query(
    query: str,
) -> list[float]:

    vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    return vector.tolist()

## Retrieval

In [33]:
def semantic_search(
    query: str,
    top_k: int = 10,
):

    query_vector = embed_query(
        query
    )

    response = (
        qdrant_client.query_points(
            collection_name=
                COLLECTION_NAME,
            query=query_vector,
            limit=top_k,
            with_payload=True,
        )
    )

    return response.points

In [34]:
query = (
    "Why does the Executive KPI Dashboard "
    "report different Net Revenue from the "
    "Finance Revenue Dashboard after April 1, 2026?"
)

results = semantic_search(
    query,
    top_k=10,
)

print(
    "Retrieved:",
    len(results)
)

Retrieved: 10


In [35]:
for rank, result in enumerate(
    results,
    start=1,
):

    payload = result.payload

    print("=" * 80)
    print(
        f"RANK: {rank}"
    )
    print(
        f"SCORE: {result.score:.4f}"
    )
    print(
        "FILE:",
        payload.get("file_name")
    )
    print(
        "ASSET:",
        payload.get("asset_type")
    )
    print(
        "METRIC:",
        payload.get("metric_name")
    )
    print(
        "VERSION:",
        payload.get(
            "observed_version"
        )
    )
    print(
        "VERSION RELATION:",
        payload.get(
            "version_relation"
        )
    )
    print()
    print(
        payload.get(
            "content",
            ""
        )[:500]
    )

RANK: 1
SCORE: 0.8037
FILE: INC-001.md
ASSET: incident
METRIC: net_revenue
VERSION: None
VERSION RELATION: unknown_observed_version

## Summary  
Finance and Executive dashboards began reporting different Net Revenue values
for reporting periods after April 1, 2026.
RANK: 2
SCORE: 0.7255
FILE: INC-001.md
ASSET: incident
METRIC: net_revenue
VERSION: None
VERSION RELATION: unknown_observed_version

## Observations  
Finance Analytics confirmed that the Finance Revenue Dashboard was migrated to
Net Revenue Version 3.  
The Executive KPI Dashboard still appears to calculate Net Revenue without
deducting posted chargebacks.
RANK: 3
SCORE: 0.6825
FILE: revenue_v3_migration.md
ASSET: analyst_note
METRIC: net_revenue
VERSION: None
VERSION RELATION: unknown_observed_version

# Revenue V3 Migration Notes  
author: Maya Sen
team: Finance Analytics
date: 2026-04-06
related_metric: net_revenue  
Finance completed the migration to Net Revenue v3 for April 2026 reporting.  
The main change from v2 is

In [36]:
retrieval_df = pd.DataFrame(
    [
        {
            "rank": rank,
            "score": result.score,
            "file_name":
                result.payload.get(
                    "file_name"
                ),
            "asset_type":
                result.payload.get(
                    "asset_type"
                ),
            "metric_name":
                result.payload.get(
                    "metric_name"
                ),
            "observed_version":
                result.payload.get(
                    "observed_version"
                ),
            "authoritative_version":
                result.payload.get(
                    "authoritative_version"
                ),
            "version_relation":
                result.payload.get(
                    "version_relation"
                ),
            "freshness_status":
                result.payload.get(
                    "freshness_status"
                ),
        }
        for rank, result in enumerate(
            results,
            start=1,
        )
    ]
)

retrieval_df

,rank,score,file_name,asset_type,metric_name,observed_version,authoritative_version,version_relation,freshness_status
0,1,0.803713,INC-001.md,incident,net_revenue,None,v3,unknown_observed_version,unknown
1,2,0.725541,INC-001.md,incident,net_revenue,None,v3,unknown_observed_version,unknown
2,3,0.682527,revenue_v3_migration.md,analyst_note,net_revenue,None,v3,unknown_observed_version,unknown
3,4,0.678258,INC-001.md,incident,net_revenue,None,v3,unknown_observed_version,unknown
4,5,0.639951,INC-001.md,incident,net_revenue,None,v3,unknown_observed_version,unknown
5,6,0.573329,INC-003.md,incident,total_orders,None,v2,unknown_observed_version,unknown
6,7,0.539322,net_revenue_v3.md,business_rule,net_revenue,v3,v3,current,unknown
7,8,0.517168,total_orders_v2.md,business_rule,total_orders,v2,v2,current,unknown
8,9,0.497113,net_revenue_v2.md,business_rule,net_revenue,v2,v3,non_current,unknown
9,10,0.466794,INC-003.md,incident,total_orders,None,v2,unknown_observed_version,unknown


In [37]:
net_revenue_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metric_name",
            match=models.MatchValue(
                value="net_revenue"
            ),
        )
    ]
)

In [38]:
filtered_response = (
    qdrant_client.query_points(
        collection_name=
            COLLECTION_NAME,
        query=embed_query(query),
        query_filter=
            net_revenue_filter,
        limit=10,
        with_payload=True,
    )
)

filtered_results = (
    filtered_response.points
)

print(
    "Filtered results:",
    len(filtered_results)
)

Filtered results: 10


In [39]:
for rank, result in enumerate(
    filtered_results,
    start=1,
):

    print(
        rank,
        round(
            result.score,
            4
        ),
        result.payload.get(
            "file_name"
        ),
        result.payload.get(
            "observed_version"
        ),
        result.payload.get(
            "version_relation"
        ),
    )

1 0.8037 INC-001.md None unknown_observed_version
2 0.7255 INC-001.md None unknown_observed_version
3 0.6825 revenue_v3_migration.md None unknown_observed_version
4 0.6783 INC-001.md None unknown_observed_version
5 0.64 INC-001.md None unknown_observed_version
6 0.5393 net_revenue_v3.md v3 current
7 0.4971 net_revenue_v2.md v2 non_current
8 0.4537 net_revenue_v1.md v1 non_current
9 0.4252 finance_revenue_dashboard.json v3 current
10 0.4218 net_revenue_v3.md v3 current


## Testing Lineage Orientation

In [40]:
lineage_query = (
    "What is the upstream lineage of "
    "Net Revenue shown on the "
    "Finance Revenue Dashboard?"
)

lineage_results = semantic_search(
    lineage_query,
    top_k=10,
)

In [41]:
for rank, result in enumerate(
    lineage_results,
    start=1,
):

    payload = result.payload

    print("=" * 70)
    print(
        rank,
        round(
            result.score,
            4
        ),
        payload.get(
            "file_name"
        )
    )
    print(
        "Lineage node:",
        payload.get(
            "lineage_node"
        )
    )
    print(
        "Upstream:",
        payload.get(
            "all_upstream"
        )
    )

1 0.6389 INC-001.md
Lineage node: None
Upstream: []
2 0.6357 revenue_v3_migration.md
Lineage node: None
Upstream: []
3 0.6024 INC-001.md
Lineage node: None
Upstream: []
4 0.5948 net_revenue_v3.md
Lineage node: None
Upstream: []
5 0.5531 net_revenue_v3.md
Lineage node: None
Upstream: []
6 0.5404 net_revenue_v1.md
Lineage node: None
Upstream: []
7 0.5389 net_revenue_v2.md
Lineage node: None
Upstream: []
8 0.5389 net_revenue_v2.md
Lineage node: None
Upstream: []
9 0.5282 INC-001.md
Lineage node: None
Upstream: []
10 0.519 net_revenue_v1.md
Lineage node: None
Upstream: []


## Creating a compact Retrieval Function

In [42]:
def retrieve_candidates(
    query: str,
    top_k: int = 20,
    query_filter=None,
) -> list[dict]:

    response = (
        qdrant_client.query_points(
            collection_name=
                COLLECTION_NAME,
            query=embed_query(query),
            query_filter=query_filter,
            limit=top_k,
            with_payload=True,
        )
    )

    return [
        {
            "point_id": str(
                point.id
            ),
            "score": float(
                point.score
            ),
            "payload":
                point.payload,
        }
        for point in response.points
    ]

In [43]:
candidates = retrieve_candidates(
    query,
    top_k=20,
)

len(candidates)

20

In [44]:
assert len(embeddings) == len(chunks)

assert (
    collection_info.points_count
    == len(chunks)
)

assert len(candidates) > 0

assert not any(
    "ground_truth"
    in candidate[
        "payload"
    ].get(
        "source_path",
        ""
    )
    for candidate in candidates
)

print(
    "✅ Phase 6.1 vector pipeline validated."
)

✅ Phase 6.1 vector pipeline validated.


# Phase 6.2 — Mandatory Cross-Encoder Reranking

Dense Qdrant retrieval is used as the fast first-stage candidate generator.

The Cross-Encoder then evaluates the user query jointly with each retrieved
candidate and produces a stronger relevance ranking.

Production retrieval target:

query
→ dense retrieval top 20
→ metadata filtering where appropriate
→ mandatory Cross-Encoder reranking
→ final top 5 evidence chunks
→ RAG

In [45]:
import torch

from sentence_transformers import CrossEncoder

In [46]:
RERANKER_MODEL = (
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

print(
    "Reranker:",
    RERANKER_MODEL
)

Reranker: cross-encoder/ms-marco-MiniLM-L6-v2


In [47]:
reranker = CrossEncoder(
    RERANKER_MODEL,
    activation_fn=torch.nn.Sigmoid(),
)

print(
    "✅ Cross-Encoder loaded."
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Cross-Encoder loaded.


In [48]:
def build_reranker_text(
    candidate: dict,
) -> str:

    payload = candidate[
        "payload"
    ]

    parts = []

    if payload.get("file_name"):
        parts.append(
            f"Source file: "
            f"{payload['file_name']}."
        )

    if payload.get("asset_type"):
        parts.append(
            f"Asset type: "
            f"{payload['asset_type']}."
        )

    if payload.get("metric_name"):
        parts.append(
            f"Metric: "
            f"{payload['metric_name']}."
        )

    if payload.get(
        "observed_version"
    ):
        parts.append(
            "Observed version: "
            f"{payload['observed_version']}."
        )

    if payload.get(
        "authoritative_version"
    ):
        parts.append(
            "Authoritative version: "
            f"{payload['authoritative_version']}."
        )

    if payload.get(
        "version_relation"
    ):
        parts.append(
            "Version relation: "
            f"{payload['version_relation']}."
        )

    if payload.get(
        "freshness_status"
    ):
        parts.append(
            "Freshness status: "
            f"{payload['freshness_status']}."
        )

    parts.append(
        payload.get(
            "content",
            ""
        )
    )

    return "\n".join(parts)

In [49]:
print(
    build_reranker_text(
        candidates[0]
    )
)

Source file: INC-001.md.
Asset type: incident.
Metric: net_revenue.
Authoritative version: v3.
Version relation: unknown_observed_version.
Freshness status: unknown.
## Summary  
Finance and Executive dashboards began reporting different Net Revenue values
for reporting periods after April 1, 2026.


In [50]:
def rerank_candidates(
    query: str,
    candidates: list[dict],
    top_k: int = 5,
) -> list[dict]:

    if not candidates:
        return []

    pairs = [
        (
            query,
            build_reranker_text(
                candidate
            ),
        )
        for candidate in candidates
    ]

    rerank_scores = reranker.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False,
    )

    reranked = []

    for candidate, score in zip(
        candidates,
        rerank_scores,
    ):

        reranked.append(
            {
                **candidate,
                "dense_score":
                    candidate["score"],
                "rerank_score":
                    float(score),
            }
        )

    reranked.sort(
        key=lambda item:
            item["rerank_score"],
        reverse=True,
    )

    return reranked[:top_k]

In [51]:
RETRIEVAL_TOP_K = 20
FINAL_TOP_K = 5

dense_candidates = retrieve_candidates(
    query,
    top_k=RETRIEVAL_TOP_K,
)

print(
    "Dense candidates:",
    len(dense_candidates)
)

Dense candidates: 20


In [52]:
for rank, candidate in enumerate(
    dense_candidates,
    start=1,
):

    candidate[
        "dense_rank"
    ] = rank

In [53]:
reranked_results = rerank_candidates(
    query=query,
    candidates=dense_candidates,
    top_k=FINAL_TOP_K,
)

print(
    "Final reranked evidence:",
    len(reranked_results)
)

Final reranked evidence: 5


In [54]:
for rank, result in enumerate(
    reranked_results,
    start=1,
):

    payload = result[
        "payload"
    ]

    print("=" * 85)

    print(
        f"RERANKED POSITION: {rank}"
    )

    print(
        "ORIGINAL DENSE RANK:",
        result.get(
            "dense_rank"
        )
    )

    print(
        "DENSE SCORE:",
        round(
            result[
                "dense_score"
            ],
            4,
        )
    )

    print(
        "RERANK SCORE:",
        round(
            result[
                "rerank_score"
            ],
            4,
        )
    )

    print(
        "FILE:",
        payload.get(
            "file_name"
        )
    )

    print(
        "METRIC:",
        payload.get(
            "metric_name"
        )
    )

    print(
        "VERSION:",
        payload.get(
            "observed_version"
        )
    )

    print(
        "RELATION:",
        payload.get(
            "version_relation"
        )
    )

    print()

    print(
        payload.get(
            "content",
            ""
        )[:600]
    )

RERANKED POSITION: 1
ORIGINAL DENSE RANK: 3
DENSE SCORE: 0.6825
RERANK SCORE: 0.9938
FILE: revenue_v3_migration.md
METRIC: net_revenue
VERSION: None
RELATION: unknown_observed_version

# Revenue V3 Migration Notes  
author: Maya Sen
team: Finance Analytics
date: 2026-04-06
related_metric: net_revenue  
Finance completed the migration to Net Revenue v3 for April 2026 reporting.  
The main change from v2 is the deduction of posted chargebacks.  
The Finance Revenue Dashboard and Finance Daily mart have been updated.  
During validation, the Executive KPI Dashboard continued to show higher Net
Revenue values for dates containing chargeback activity.  
The Executive pipeline appears to remain on v2 and should be reviewed by the
BI Platform team.  
Until migration is complete, Fi
RERANKED POSITION: 2
ORIGINAL DENSE RANK: 1
DENSE SCORE: 0.8037
RERANK SCORE: 0.9908
FILE: INC-001.md
METRIC: net_revenue
VERSION: None
RELATION: unknown_observed_version

## Summary  
Finance and Executive dashboa

In [55]:
rerank_comparison = pd.DataFrame(
    [
        {
            "rerank_rank": rank,
            "dense_rank":
                result.get(
                    "dense_rank"
                ),
            "dense_score":
                result[
                    "dense_score"
                ],
            "rerank_score":
                result[
                    "rerank_score"
                ],
            "file_name":
                result[
                    "payload"
                ].get(
                    "file_name"
                ),
            "asset_type":
                result[
                    "payload"
                ].get(
                    "asset_type"
                ),
            "metric_name":
                result[
                    "payload"
                ].get(
                    "metric_name"
                ),
            "observed_version":
                result[
                    "payload"
                ].get(
                    "observed_version"
                ),
            "version_relation":
                result[
                    "payload"
                ].get(
                    "version_relation"
                ),
        }
        for rank, result in enumerate(
            reranked_results,
            start=1,
        )
    ]
)

rerank_comparison

,rerank_rank,dense_rank,dense_score,rerank_score,file_name,asset_type,metric_name,observed_version,version_relation
0,1,3,0.682527,0.993834,revenue_v3_migration.md,analyst_note,net_revenue,None,unknown_observed_version
1,2,1,0.803713,0.990768,INC-001.md,incident,net_revenue,None,unknown_observed_version
2,3,2,0.725541,0.954850,INC-001.md,incident,net_revenue,None,unknown_observed_version
3,4,5,0.639951,0.216452,INC-001.md,incident,net_revenue,None,unknown_observed_version
4,5,7,0.539322,0.084064,net_revenue_v3.md,business_rule,net_revenue,v3,current


In [56]:
def retrieve_and_rerank(
    query: str,
    retrieval_top_k: int = 20,
    final_top_k: int = 5,
    query_filter=None,
) -> list[dict]:

    candidates = retrieve_candidates(
        query=query,
        top_k=retrieval_top_k,
        query_filter=query_filter,
    )

    for rank, candidate in enumerate(
        candidates,
        start=1,
    ):
        candidate[
            "dense_rank"
        ] = rank

    return rerank_candidates(
        query=query,
        candidates=candidates,
        top_k=final_top_k,
    )

In [57]:
final_evidence = retrieve_and_rerank(
    query
)

len(final_evidence)

5

In [58]:
filtered_reranked = (
    retrieve_and_rerank(
        query=query,
        retrieval_top_k=20,
        final_top_k=5,
        query_filter=
            net_revenue_filter,
    )
)

len(filtered_reranked)

5

In [59]:
assert all(
    result[
        "payload"
    ].get(
        "metric_name"
    )
    == "net_revenue"

    for result
    in filtered_reranked
)

print(
    "✅ Metadata filter preserved through reranking."
)

✅ Metadata filter preserved through reranking.


### Test

In [60]:
active_customer_query = (
    "Why does the Growth and Marketing "
    "Dashboard report more Active Customers "
    "than the enterprise KPI?"
)

In [61]:
active_customer_results = (
    retrieve_and_rerank(
        query=active_customer_query,
        retrieval_top_k=20,
        final_top_k=5,
    )
)

In [62]:
for rank, result in enumerate(
    active_customer_results,
    start=1,
):

    payload = result[
        "payload"
    ]

    print(
        rank,
        "| dense:",
        result.get(
            "dense_rank"
        ),
        "| rerank:",
        round(
            result[
                "rerank_score"
            ],
            4,
        ),
        "|",
        payload.get(
            "file_name"
        ),
        "|",
        payload.get(
            "metric_name"
        ),
    )

1 | dense: 2 | rerank: 0.9919 | INC-002.md | active_customers
2 | dense: 1 | rerank: 0.9465 | active_customer_review.md | active_customers
3 | dense: 3 | rerank: 0.8678 | INC-002.md | active_customers
4 | dense: 6 | rerank: 0.4319 | INC-002.md | active_customers
5 | dense: 7 | rerank: 0.3213 | active_customers_v2.md | active_customers


In [63]:
orders_query = (
    "Is the Total Orders disagreement "
    "between Operations and Finance "
    "a data pipeline failure?"
)

orders_results = (
    retrieve_and_rerank(
        query=orders_query,
        retrieval_top_k=20,
        final_top_k=5,
    )
)

In [64]:
for rank, result in enumerate(
    orders_results,
    start=1,
):

    payload = result[
        "payload"
    ]

    print(
        rank,
        "| dense:",
        result.get(
            "dense_rank"
        ),
        "| rerank:",
        round(
            result[
                "rerank_score"
            ],
            4,
        ),
        "|",
        payload.get(
            "file_name"
        ),
    )

1 | dense: 1 | rerank: 0.9321 | total_orders_semantics.md
2 | dense: 12 | rerank: 0.6631 | INC-003.md
3 | dense: 2 | rerank: 0.1548 | INC-003.md
4 | dense: 7 | rerank: 0.117 | INC-003.md
5 | dense: 3 | rerank: 0.0935 | INC-003.md


## Measuring Rank Movement

In [65]:
def build_rank_movement_table(
    results: list[dict],
) -> pd.DataFrame:

    rows = []

    for rerank_rank, result in enumerate(
        results,
        start=1,
    ):

        dense_rank = result.get(
            "dense_rank"
        )

        rows.append(
            {
                "file_name":
                    result[
                        "payload"
                    ].get(
                        "file_name"
                    ),
                "dense_rank":
                    dense_rank,
                "rerank_rank":
                    rerank_rank,
                "rank_improvement":
                    (
                        dense_rank
                        - rerank_rank
                        if dense_rank
                        is not None
                        else None
                    ),
                "rerank_score":
                    result[
                        "rerank_score"
                    ],
            }
        )

    return pd.DataFrame(
        rows
    )

In [66]:
build_rank_movement_table(
    reranked_results
)

,file_name,dense_rank,rerank_rank,rank_improvement,rerank_score
0,revenue_v3_migration.md,3,1,2,0.993834
1,INC-001.md,1,2,-1,0.990768
2,INC-001.md,2,3,-1,0.954850
3,INC-001.md,5,4,1,0.216452
4,net_revenue_v3.md,7,5,2,0.084064


## Creating final evidence formatting

In [67]:
def format_final_evidence(
    results: list[dict],
) -> list[dict]:

    evidence = []

    for rank, result in enumerate(
        results,
        start=1,
    ):

        payload = result[
            "payload"
        ]

        evidence.append(
            {
                "evidence_rank":
                    rank,
                "rerank_score":
                    result[
                        "rerank_score"
                    ],
                "source_path":
                    payload.get(
                        "source_path"
                    ),
                "file_name":
                    payload.get(
                        "file_name"
                    ),
                "asset_type":
                    payload.get(
                        "asset_type"
                    ),
                "metric_name":
                    payload.get(
                        "metric_name"
                    ),
                "observed_version":
                    payload.get(
                        "observed_version"
                    ),
                "authoritative_version":
                    payload.get(
                        "authoritative_version"
                    ),
                "version_relation":
                    payload.get(
                        "version_relation"
                    ),
                "freshness_status":
                    payload.get(
                        "freshness_status"
                    ),
                "lineage_node":
                    payload.get(
                        "lineage_node"
                    ),
                "content":
                    payload.get(
                        "content"
                    ),
            }
        )

    return evidence

In [68]:
revenue_evidence = (
    format_final_evidence(
        reranked_results
    )
)

revenue_evidence

[{'evidence_rank': 1,
  'rerank_score': 0.9938342571258545,
  'source_path': 'data/raw/analyst_notes/revenue_v3_migration.md',
  'file_name': 'revenue_v3_migration.md',
  'asset_type': 'analyst_note',
  'metric_name': 'net_revenue',
  'observed_version': None,
  'authoritative_version': 'v3',
  'version_relation': 'unknown_observed_version',
  'freshness_status': 'unknown',
  'lineage_node': None,
  'content': '# Revenue V3 Migration Notes  \nauthor: Maya Sen\nteam: Finance Analytics\ndate: 2026-04-06\nrelated_metric: net_revenue  \nFinance completed the migration to Net Revenue v3 for April 2026 reporting.  \nThe main change from v2 is the deduction of posted chargebacks.  \nThe Finance Revenue Dashboard and Finance Daily mart have been updated.  \nDuring validation, the Executive KPI Dashboard continued to show higher Net\nRevenue values for dates containing chargeback activity.  \nThe Executive pipeline appears to remain on v2 and should be reviewed by the\nBI Platform team.  \nUnti

In [69]:
assert len(
    reranked_results
) == FINAL_TOP_K

assert all(
    "rerank_score"
    in result
    for result in reranked_results
)

assert all(
    result[
        "payload"
    ].get(
        "source_path"
    )
    for result in reranked_results
)

assert not any(
    "ground_truth"
    in result[
        "payload"
    ].get(
        "source_path",
        "",
    )
    for result in reranked_results
)

print(
    "✅ Mandatory reranking pipeline validated."
)

✅ Mandatory reranking pipeline validated.


## Phase 6.1 Summary

MetricGuard now has a working dense semantic retrieval prototype.

### Completed

- Regenerated fully enriched retrieval chunks using production code.
- Loaded a Sentence Transformers dense embedding model.
- Generated one embedding per retrieval chunk.
- Validated embedding dimensions and numeric integrity.
- Created a local Qdrant collection.
- Stored vectors with full MetricGuard payload metadata.
- Implemented semantic nearest-neighbor retrieval.
- Demonstrated metadata-filtered retrieval.
- Demonstrated lineage-aware retrieval.
- Confirmed ground-truth evaluation files were excluded.

### Retrieval architecture

User query
→ query embedding
→ Qdrant dense retrieval
→ metadata filtering
→ candidate evidence

### Next

Mandatory Cross-Encoder reranking will reorder the retrieved candidate set
before evidence is passed into baseline RAG.

## Phase 6.2 Summary — Reranking

MetricGuard now uses a two-stage evidence retrieval architecture.

### Stage 1 — Dense Retrieval

The user query is embedded using the same bi-encoder used for knowledge chunks.

Qdrant retrieves a relatively broad candidate set based on semantic vector
similarity and optional metadata filtering.

### Stage 2 — Cross-Encoder Reranking

The Cross-Encoder receives the user query and each candidate jointly.

The candidates are scored again for query-document relevance and reordered.

Current prototype:

- Dense retrieval: Top 20
- Mandatory Cross-Encoder reranking
- Final evidence: Top 5
- Reranker: `cross-encoder/ms-marco-MiniLM-L6-v2`

### Final Retrieval Flow

query
→ query embedding
→ Qdrant
→ top 20 candidates
→ metadata filtering where appropriate
→ mandatory Cross-Encoder reranking
→ top 5 evidence chunks
→ baseline RAG

### Design Principle

Dense similarity scores and reranker scores are produced by different models
and are not compared numerically.

Dense retrieval is responsible for recall.

The Cross-Encoder is responsible for improving precision within the retrieved
candidate set.

### Next

Move the retrieval + reranking logic into the production MetricGuard package,
then build baseline RAG using only the final reranked evidence.

## Phase 6.3 — Production Retrieval Integration

The retrieval and reranking prototype has now been moved into:

`src/metricguard/retrieval/`

This section validates the production package against the real Sentence
Transformer embedding model, Qdrant collection, and Cross-Encoder reranker.

In [70]:
import subprocess

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "pull",
        "origin",
        "main",
    ],
    check=True,
)

CompletedProcess(args=['git', '-C', '/content/metricguard-ai', 'pull', 'origin', 'main'], returncode=0)

In [71]:
%pip install -q -e "/content/metricguard-ai"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricguard-ai (pyproject.toml) ... done


In [72]:
from metricguard.retrieval import (
    CrossEncoderReranker,
    DenseRetriever,
    RetrievalPipeline,
    format_final_evidence,
    load_retrieval_config,
)

In [73]:
retrieval_config = (
    load_retrieval_config(
        REPO_DIR
    )
)

retrieval_config

RetrievalConfig(candidate_top_k=20, final_top_k=5, reranker_model='cross-encoder/ms-marco-MiniLM-L6-v2', embedding_model='sentence-transformers/all-mpnet-base-v2', collection_name='metricguard_dense_v1', normalize_embeddings=True)

In [74]:
production_dense_retriever = (
    DenseRetriever(
        client=qdrant_client,
        embedding_model=
            embedding_model,
        collection_name=
            retrieval_config
            .collection_name,
        normalize_embeddings=
            retrieval_config
            .normalize_embeddings,
    )
)

In [75]:
production_reranker = (
    CrossEncoderReranker(
        model=reranker,
        batch_size=16,
    )
)

In [76]:
production_retrieval = (
    RetrievalPipeline(
        dense_retriever=
            production_dense_retriever,
        reranker=
            production_reranker,
        candidate_top_k=
            retrieval_config
            .candidate_top_k,
        final_top_k=
            retrieval_config
            .final_top_k,
    )
)

In [77]:
production_results = (
    production_retrieval.retrieve(
        query
    )
)

len(production_results)

5

In [78]:
for result in production_results:

    print(
        "RERANK:",
        result.rerank_rank,
        "| DENSE:",
        result.dense_rank,
        "| SCORE:",
        round(
            result.rerank_score,
            4,
        ),
        "| FILE:",
        result.payload.get(
            "file_name"
        ),
    )

RERANK: 1 | DENSE: 3 | SCORE: 0.9938 | FILE: revenue_v3_migration.md
RERANK: 2 | DENSE: 1 | SCORE: 0.9908 | FILE: INC-001.md
RERANK: 3 | DENSE: 2 | SCORE: 0.9549 | FILE: INC-001.md
RERANK: 4 | DENSE: 5 | SCORE: 0.2165 | FILE: INC-001.md
RERANK: 5 | DENSE: 7 | SCORE: 0.0841 | FILE: net_revenue_v3.md


In [79]:
production_evidence = (
    format_final_evidence(
        production_results
    )
)

production_evidence

[{'evidence_rank': 1,
  'dense_rank': 3,
  'dense_score': 0.682527076155074,
  'rerank_score': 0.9938342571258545,
  'source_path': 'data/raw/analyst_notes/revenue_v3_migration.md',
  'file_name': 'revenue_v3_migration.md',
  'asset_type': 'analyst_note',
  'metric_name': 'net_revenue',
  'observed_version': None,
  'authoritative_version': 'v3',
  'version_relation': 'unknown_observed_version',
  'freshness_status': 'unknown',
  'lineage_node': None,
  'direct_upstream': [],
  'all_upstream': [],
  'direct_downstream': [],
  'all_downstream': [],
  'content': '# Revenue V3 Migration Notes  \nauthor: Maya Sen\nteam: Finance Analytics\ndate: 2026-04-06\nrelated_metric: net_revenue  \nFinance completed the migration to Net Revenue v3 for April 2026 reporting.  \nThe main change from v2 is the deduction of posted chargebacks.  \nThe Finance Revenue Dashboard and Finance Daily mart have been updated.  \nDuring validation, the Executive KPI Dashboard continued to show higher Net\nRevenue va

In [81]:
assert len(
    production_results
) == 5

assert all(
    result.rerank_rank
    >= 1
    for result
    in production_results
)

assert all(
    result.payload.get(
        "source_path"
    )
    for result
    in production_results
)

assert not any(
    "ground_truth"
    in result.payload.get(
        "source_path",
        "",
    )
    for result
    in production_results
)

print(
    "✅ Production retrieval + "
    "Reranking validated."
)

✅ Production retrieval + Reranking validated.


### Phase 6.3 Summary — Production Retrieval

The dense retrieval and mandatory reranking prototype has been productionized
into `src/metricguard/retrieval/`.

### Production modules

- configuration loader
- retrieval data contracts
- dense Qdrant retriever
- Cross-Encoder reranker
- final retrieval pipeline
- final evidence formatter
- lazy model factories

### Runtime architecture

query
→ Sentence Transformer embedding
→ Qdrant top-20 retrieval
→ optional metadata filtering
→ Cross-Encoder reranking
→ top-5 final evidence
→ provenance validation
→ RAG-ready evidence package

### Testing strategy

Local tests use lightweight fake embedding, Qdrant, and Cross-Encoder
components.

Real model integration is validated in Colab against the actual MetricGuard
Qdrant collection.

### Next

Build baseline RAG over the final reranked evidence.